In [18]:
import pandas as pd
from prophet import Prophet
from dotenv import load_dotenv
import plotly.express as px
import os
import utils
load_dotenv()

True

In [5]:
os.getenv('DATA_STORAGE')

'../../../data/csv/'

In [3]:
df_rte = pd.read_csv(os.getenv('DATA_STORAGE') + 'power_consumption_meteo_2021_2025_utc.csv')
df_rte.head()

,start_date,end_date,value,mean,min,max,median,std,q1,q3
0,2021-01-01 00:00:00,2021-01-01 01:00:00,63955.5,1.636,-2.5,9.9,1.0,2.849473,-0.575,2.975
1,2021-01-01 01:00:00,2021-01-01 02:00:00,62054.0,1.340,-3.5,10.2,1.0,3.125145,-1.500,3.300
2,2021-01-01 02:00:00,2021-01-01 03:00:00,59021.5,1.092,-3.1,10.0,0.8,3.180864,-2.000,2.875
3,2021-01-01 03:00:00,2021-01-01 04:00:00,57391.5,0.966,-2.8,9.2,0.5,3.160574,-1.775,2.875
4,2021-01-01 04:00:00,2021-01-01 05:00:00,57620.5,0.802,-2.8,9.6,0.0,3.094530,-1.675,2.650


In [4]:
dfp = utils.convert_df_to_prophet(df_rte)
dfp.head()

,ds,y,mean,min,max,median,std,q1,q3
0,2021-01-01 01:00:00,63955.5,1.636,-2.5,9.9,1.0,2.849473,-0.575,2.975
1,2021-01-01 02:00:00,62054.0,1.340,-3.5,10.2,1.0,3.125145,-1.500,3.300
2,2021-01-01 03:00:00,59021.5,1.092,-3.1,10.0,0.8,3.180864,-2.000,2.875
3,2021-01-01 04:00:00,57391.5,0.966,-2.8,9.2,0.5,3.160574,-1.775,2.875
4,2021-01-01 05:00:00,57620.5,0.802,-2.8,9.6,0.0,3.094530,-1.675,2.650


In [12]:
dataset = dfp.copy()
dataset = dataset[['ds', 'y', 'mean']]
train_set = dataset[ (dataset['ds'] >= '2025-09-01') & (dataset['ds'] < '2025-10-01') ]
test_set = dataset[ (dataset['ds'] >= '2025-10-01') & (dataset['ds'] < '2025-10-02') ]
test_set.head()

,ds,y,mean
41611,2025-10-01 00:00:00,40653.0,11.490
41612,2025-10-01 01:00:00,39416.5,11.040
41613,2025-10-01 02:00:00,37804.0,10.390
41614,2025-10-01 03:00:00,37189.5,9.956
41615,2025-10-01 04:00:00,39027.5,9.636


In [13]:
m_reg = Prophet()
#m_reg.add_regressor('mean')
m_reg.fit(train_set)

08:48:52 - cmdstanpy - INFO - Chain [1] start processing
08:48:52 - cmdstanpy - INFO - Chain [1] done processing


In [15]:
df_pred = m_reg.predict(test_set)
df_pred.head()

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,daily,daily_lower,daily_upper,weekly,weekly_lower,weekly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2025-10-01 00:00:00,44789.743115,39469.488458,43112.183418,44789.743115,44789.743115,-3456.847880,-3456.847880,-3456.847880,-4729.805415,-4729.805415,-4729.805415,1272.957534,1272.957534,1272.957534,0.0,0.0,0.0,41332.895234
1,2025-10-01 01:00:00,44786.860840,37163.176031,40813.373993,44786.860840,44786.860840,-5687.336226,-5687.336226,-5687.336226,-6990.770939,-6990.770939,-6990.770939,1303.434713,1303.434713,1303.434713,0.0,0.0,0.0,39099.524614
2,2025-10-01 02:00:00,44783.978565,35750.400686,39315.851173,44783.978565,44783.978565,-7328.546790,-7328.546790,-7328.546790,-8665.925748,-8665.925748,-8665.925748,1337.378958,1337.378958,1337.378958,0.0,0.0,0.0,37455.431775
3,2025-10-01 03:00:00,44781.096290,35644.143266,39260.274248,44781.096290,44781.096290,-7343.331562,-7343.331562,-7343.331562,-8717.683422,-8717.683422,-8717.683422,1374.351860,1374.351860,1374.351860,0.0,0.0,0.0,37437.764728
4,2025-10-01 04:00:00,44778.214015,37593.281818,41199.558773,44778.214015,44778.214015,-5417.761989,-5417.761989,-5417.761989,-6831.643559,-6831.643559,-6831.643559,1413.881570,1413.881570,1413.881570,0.0,0.0,0.0,39360.452026


In [22]:
display_mask = (dataset['ds'] >= '2025-09-30') & (dataset['ds'] < '2025-10-03')
fig = px.line(dataset[display_mask], x='ds', y='y', range_y=[0, None], title='Test')
fig.add_trace(px.scatter(df_pred, x='ds', y='yhat').data[0])
fig.data[1].marker = dict(color='red')
fig.show()

In [23]:
m_reg2 = Prophet()
m_reg2.add_regressor('mean')
m_reg2.fit(train_set)
df_pred2 = m_reg2.predict(test_set)
df_pred2.head()

08:57:28 - cmdstanpy - INFO - Chain [1] start processing
08:57:28 - cmdstanpy - INFO - Chain [1] done processing


,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,daily,...,mean,mean_lower,mean_upper,weekly,weekly_lower,weekly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2025-10-01 00:00:00,44898.013833,39378.633822,43213.404417,44898.013833,44898.013833,-3505.958428,-3505.958428,-3505.958428,-4696.109674,...,-98.386811,-98.386811,-98.386811,1288.538058,1288.538058,1288.538058,0.0,0.0,0.0,41392.055405
1,2025-10-01 01:00:00,44895.722752,37297.219379,41044.082048,44895.722752,44895.722752,-5737.003404,-5737.003404,-5737.003404,-6948.529706,...,-107.505713,-107.505713,-107.505713,1319.032014,1319.032014,1319.032014,0.0,0.0,0.0,39158.719347
2,2025-10-01 02:00:00,44893.431670,35629.085796,39335.752118,44893.431670,44893.431670,-7385.012171,-7385.012171,-7385.012171,-8617.307246,...,-120.677459,-120.677459,-120.677459,1352.972535,1352.972535,1352.972535,0.0,0.0,0.0,37508.419500
3,2025-10-01 03:00:00,44891.140589,35503.136927,39457.736728,44891.140589,44891.140589,-7403.478874,-7403.478874,-7403.478874,-8663.938008,...,-129.472133,-129.472133,-129.472133,1389.931267,1389.931267,1389.931267,0.0,0.0,0.0,37487.661714
4,2025-10-01 04:00:00,44888.849507,37647.549873,41186.842050,44888.849507,44888.849507,-5478.586276,-5478.586276,-5478.586276,-6772.075877,...,-135.956685,-135.956685,-135.956685,1429.446286,1429.446286,1429.446286,0.0,0.0,0.0,39410.263231


In [24]:
fig2 = px.line(dataset[display_mask], x='ds', y='y', range_y=[0, None], title='Test')
fig2.add_trace(px.scatter(df_pred2, x='ds', y='yhat').data[0])
fig2.data[1].marker = dict(color='red')
fig2.show()

--------------------------------------------------------------------

In [4]:
dataset = dfp.copy()
dataset = dataset[dataset['ds'] < '2025-10-01']
train_period = "30 days"
test_period = "1 day"
period = "53 hours"
regressors = ['mean']
title = f"train_period: {train_period} - test_period: {test_period} - period: {period} - regressors: {",".join(regressors)}"
train_test_sets = utils.create_train_test(dataset, train_period, test_period, period=period, n_iter=5)

In [5]:
train_test_sets[0][0].head()

,ds,y,mean,min,max,median,std,q1,q3
40654,2025-08-22 03:00:00,33385.0,15.310,12.1,20.8,14.80,2.346535,13.400,16.800
40655,2025-08-22 04:00:00,34677.5,14.966,12.0,20.7,14.10,2.457110,12.925,16.725
40656,2025-08-22 05:00:00,36931.0,14.646,11.5,20.5,13.80,2.626724,12.400,16.475
40657,2025-08-22 06:00:00,39267.5,14.430,11.1,20.1,13.80,2.686113,11.925,16.175
40658,2025-08-22 07:00:00,41943.0,14.820,11.9,20.8,14.05,2.681836,12.500,16.325


In [6]:
results = utils.train_predict(train_test_sets, regressors=regressors)

17:54:30 - cmdstanpy - INFO - Chain [1] start processing
17:54:30 - cmdstanpy - INFO - Chain [1] done processing


SET 0
train from 2025-08-22 03:00:00 to 2025-09-21 02:00:00
test from 2025-09-21 03:00:00 to 2025-09-22 02:00:00
columns Index(['ds', 'y', 'mean', 'min', 'max', 'median', 'std', 'q1', 'q3'], dtype='object')


ValueError: Regressor 'mean' missing from dataframe